# 異種モデルの分類制約付きベイズ最適化：単目的・多目的回帰 × 多クラス・順序分類

単目的回帰 × 多クラス分類、単目的回帰 × 順序分類、多目的回帰 × 多クラス分類、多目的回帰 × 順序分類の4例を、モデル作成から候補提案まで順に実行します。

分類モデルそのものの学習・Active Learningは[30番](30_multiclass_classification_active_learning.ipynb)と[31番](31_ordinal_classification_active_learning.ipynb)を参照してください。

[理論](../../docs/theory/30_heterogeneous_classification_constraints.md) / [実装ガイド](../../docs/guide/30_heterogeneous_classification_constraints.md)

## 1. 合成データを用意する

製造条件 `x` に対する連続目的 `score` と、3段階の品質ラベル `quality` を用意します。ラベルは **0=不良、1=要確認、2=良好** と解釈します。許容するのはクラス1と2です。

同じラベルを、順序を仮定しない多クラスGPと、順序を仮定するOrdinal GPでそれぞれ学習し、違いを比較します。

In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.acquisition import make_classification_constrained_qei_acquisition
from robotorchan.models.classification.multiclass.standard.single_task import MulticlassSingleTaskGPClassifier
from robotorchan.models.classification.ordinal.standard.single_task import OrdinalSingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim import optimize_acqf
from robotorchan.semantics import ClassificationConstraint, ProblemSemantics, RegressionObjective

torch.manual_seed(11)
train_X = torch.linspace(0, 1, 15, dtype=torch.double).unsqueeze(-1)
train_Y = 1.0 - (train_X - 0.72).square()
train_quality = torch.bucketize(train_X.squeeze(-1), torch.tensor([0.34, 0.67], dtype=torch.double))
print("各クラスの件数:", torch.bincount(train_quality, minlength=3).tolist())
regression = SingleTaskGP(train_X, train_Y)
multiclass = MulticlassSingleTaskGPClassifier(train_X, train_quality, num_classes=3)
ordinal = OrdinalSingleTaskGPClassifier(train_X, train_quality, num_classes=3)

## 2. 多クラスGPと順序GPを学習する

多クラス分類はクラスを区別しますが、クラス間の順序を必ずしも仮定しません。一方、順序分類では潜在値と閾値を使って順序付きカテゴリの確率を表現します。

いずれも `predict_proba(X)` の最後の次元が3クラスの確率です。ここでは説明用に短めの学習を行います。結果の精度評価を目的とした設定ではありません。

In [ ]:
multiclass.fit(epochs=20, lr=0.03)
ordinal.fit(epochs=20, lr=0.03)
grid_X = torch.linspace(0, 1, 21, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    multi_probs = multiclass.predict_proba(grid_X)
    ordinal_probs = ordinal.predict_proba(grid_X)
print("多クラスの確率形状:", tuple(multi_probs.shape))
print("順序分類の確率形状:", tuple(ordinal_probs.shape))
assert multi_probs.shape == ordinal_probs.shape == (21, 3)

## 3. 複数クラスを許容する確率を比較する

`feasible_classes=(1, 2)` は、要確認または良好の確率を足す指定です。

$$
P(C(x)\\in\\{1,2\\}\\mid D)=P(C(x)=1\\mid D)+P(C(x)=2\\mid D)
$$

順序GPでも許容確率の計算は同じです。ただし、クラス確率自体はモデルの仮定によって変わります。

In [ ]:
allowed_classes = (1, 2)
with torch.no_grad():
    multi_allowed = multi_probs[..., list(allowed_classes)].sum(dim=-1)
    ordinal_allowed = ordinal_probs[..., list(allowed_classes)].sum(dim=-1)
print("x=0.5 付近の許容確率（多クラス）:", multi_allowed[10].item())
print("x=0.5 付近の許容確率（順序）:", ordinal_allowed[10].item())

## 4. モデルごとに回帰qEI×許容確率を構築する

`HeterogeneousModel` で回帰GPと分類GPを組み合わせます。クラス集合の指定は同一ですが、分類モデルが違うため獲得関数の形状や候補は異なり得ます。

この確率重み付けはソフト制約であり、例えば「許容確率80%以上」というハード制約の保証ではありません。

In [ ]:
semantics = ProblemSemantics(
    objectives=(RegressionObjective("score"),),
    constraints=(ClassificationConstraint("quality", feasible_classes=allowed_classes),),
)
bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)
results = {}
for label, classifier in [("multiclass", multiclass), ("ordinal", ordinal)]:
    model = HeterogeneousModel(regression, classifier, output_names=["score", "quality"])
    acquisition = make_classification_constrained_qei_acquisition(
        model,
        semantics,
        best_f=train_Y.max().item(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=11),
    )
    candidate, value = optimize_acqf(
        acquisition, bounds, q=1, optimizer="sobol", raw_samples=128, seed=11
    )
    with torch.no_grad():
        allowed_probability = classifier.predict_proba(candidate)[..., list(allowed_classes)].sum(dim=-1)
    results[label] = (candidate, value, allowed_probability)
    print(label, "候補:", candidate.flatten().tolist(), "獲得値:", value, "許容確率:", allowed_probability)
    assert candidate.shape == (1, 1)

## 5. 結果の読み方と拡張

- 多クラスGPと順序GPの提案候補が一致する必要はありません。学習されたクラス確率とその構造が異なるためです。
- `feasible_classes=(2,)` なら良好のみ、`(1,2)` なら要確認以上を許容する設計です。何を許容するかは業務上の判断です。
- q>1は候補間の許容確率を積で近似します。分類器の候補間相関を考慮した同時確率ではありません。
- `probability_threshold` を付けたハード制約は、この獲得関数ビルダーでは非対応です。
- Mixed変数は対応Optimizerと適切なモデル設計が必要です。今回のNotebookは連続1変数の候補提案を示します。

参考文献と式の前提は[理論章](../../docs/theory/30_heterogeneous_classification_constraints.md)を参照してください。

---

# 第2部：多目的回帰 × 分類制約

以下では、目的値を2つに増やし、qEHVIと分類確率の重み付けを組み合わせます。単目的のqEIと異なり、2つの目的のPareto改善を評価します。

## 1. 二目的の回帰データと3クラスのラベル

2目的は異なる最適点を持つため、両方を同時に最大化するとトレードオフが生じます。分類ラベルは `0=不良`、`1=要確認`、`2=良好` とします。

In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning

from robotorchan.acquisition import make_constrained_qehvi_acquisition
from robotorchan.models.classification.multiclass.standard.single_task import MulticlassSingleTaskGPClassifier
from robotorchan.models.classification.ordinal.standard.single_task import OrdinalSingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim import optimize_acqf
from robotorchan.semantics import ClassificationConstraint, ProblemSemantics, RegressionObjective

torch.manual_seed(17)
train_X = torch.linspace(0, 1, 15, dtype=torch.double).unsqueeze(-1)
y1 = 1.0 - (train_X - 0.25).square()
y2 = 1.0 - (train_X - 0.78).square()
train_Y = torch.cat([y1, y2], dim=-1)
train_C = torch.bucketize(train_X.squeeze(-1), torch.tensor([0.34, 0.67], dtype=torch.double))
regression = SingleTaskGP(train_X, train_Y)
multiclass = MulticlassSingleTaskGPClassifier(train_X, train_C, num_classes=3)
ordinal = OrdinalSingleTaskGPClassifier(train_X, train_C, num_classes=3)
print("回帰出力:", tuple(train_Y.shape), "分類ラベル:", torch.bincount(train_C, minlength=3).tolist())

## 2. 分類モデルの学習と参照点

`NondominatedPartitioning` は多目的獲得関数が改善する領域を計算するためのBoTorch標準部品です。参照点は、ここでは観測目的値より十分小さい `(0, 0)` を採用します。実際の問題では、目的のスケールと意味に合わせて選んでください。

In [ ]:
multiclass.fit(epochs=20, lr=0.03)
ordinal.fit(epochs=20, lr=0.03)
ref_point = torch.tensor([0.0, 0.0], dtype=torch.double)
partitioning = NondominatedPartitioning(ref_point=ref_point, Y=train_Y)
semantics = ProblemSemantics(
    objectives=(RegressionObjective("objective_1"), RegressionObjective("objective_2")),
    constraints=(ClassificationConstraint("quality", feasible_classes=(1, 2)),),
)
bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)

## 3. 多目的回帰 × 多クラス分類

`make_constrained_qehvi_acquisition` は回帰モデルの2出力からqEHVIを構築し、分類の許容クラス確率を掛け合わせます。次の候補をSobolで探索します。

In [ ]:
heterogeneous_multi = HeterogeneousModel(
    regression, multiclass,
    output_names=["objective_1", "objective_2", "quality"],
)
acq_multi = make_constrained_qehvi_acquisition(
    heterogeneous_multi, semantics, ref_point=ref_point,
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=17),
)
candidate_multi, score_multi = optimize_acqf(
    acq_multi, bounds, q=1, optimizer="sobol", raw_samples=128, seed=17
)
print("多目的×多クラスの候補:", candidate_multi, "獲得関数値:", score_multi)
print("許容確率:", multiclass.predict_proba(candidate_multi)[..., [1, 2]].sum(-1))
assert candidate_multi.shape == (1, 1)

## 4. 多目的回帰 × 順序分類

回帰モデル、2つの目的、参照点、Optimizerは同じまま、分類モデルだけをOrdinal GPに交換します。順序モデルの仮定によって許容確率が変わり、提案候補も変わり得ます。

In [ ]:
heterogeneous_ordinal = HeterogeneousModel(
    regression, ordinal,
    output_names=["objective_1", "objective_2", "quality"],
)
acq_ordinal = make_constrained_qehvi_acquisition(
    heterogeneous_ordinal, semantics, ref_point=ref_point,
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32]), seed=17),
)
candidate_ordinal, score_ordinal = optimize_acqf(
    acq_ordinal, bounds, q=1, optimizer="sobol", raw_samples=128, seed=17
)
print("多目的×順序分類の候補:", candidate_ordinal, "獲得関数値:", score_ordinal)
print("許容確率:", ordinal.predict_proba(candidate_ordinal)[..., [1, 2]].sum(-1))
assert candidate_ordinal.shape == (1, 1)

## 5. 結果の解釈

このNotebookは **多目的回帰 × 多クラス分類** と **多目的回帰 × 順序分類** をそれぞれモデル構築から候補提案まで示します。qEHVIは2目的の改善を評価し、分類器の許容確率がそのスコアを重み付けします。

`q>1` では分類側の確率積に独立性の近似が入ります。`probability_threshold` を使う厳密な閾値制約は扱いません。実験を実施した後は新たな2目的値と分類ラベルでモデルを更新してください。